In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]
    

In [ ]:
llm = ChatOllama(model="llama3")


In [ ]:
def chatNode(state:ChatState):
    
    messages=state['messages']
    
    response=llm.invoke(messages)
    
    return {'messages':[response]}



In [ ]:
graph=StateGraph(ChatState)

graph.add_node('chatNode',chatNode)

graph.add_edge(START,'chatNode')

graph.add_edge('chatNode',END)



In [ ]:
checkpointer=MemorySaver()

chatBot=graph.compile(checkpointer=checkpointer)

chatBot

In [ ]:
threadId="1"
while True:
    userMessage=input("Type Here    :   ")
    
    print(f"User Msg :   {userMessage}")
    if userMessage.strip().lower() in ['exit','quit','bye']:
        break
    
    config={"configurable":{"thread_id":threadId}}
    response=chatBot.invoke({'messages' : [HumanMessage(content=userMessage)]},config=config)
    
    print(f"AI Msg   :   {response['messages'][-1].content}")
    
    
    
    